[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/bsheese/125/blob/main/08_data_vis/08_2_Choosing_a_Chart.ipynb)

# 08.2: Choosing a Chart

Notebook 08.1 showed how to draw some charts, and this notebook is about which one to draw. Python will draw almost anything you ask for, whether or not it makes sense, so the choice is up to you. We add two chart types for looking at relationships and change over time, look at correlation, and then practice matching a chart to a question, describing what it shows and noticing when a chart misleads.

This notebook opens in Google Colab. If Colab warns that the notebook was not authored by Google, click **Run anyway**. Your changes are not kept when you close the tab unless you choose **File > Save a copy in Drive**, so do that if you want to keep your work.

## The data

We use the crime data from 08.1.

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

crime = pd.read_csv("https://raw.githubusercontent.com/bsheese/CSDS125ExampleData/master/data_corgis_state_crime.csv")
crime = crime.rename(columns={"Rates.Violent.All": "Violent", "Rates.Property.All": "Property"})
crime = crime[["State", "Year", "Population", "Violent", "Property"]]
places_2012 = crime[(crime["Year"] == 2012) & (crime["State"] != "United States")]
places_2012.head(3)

## Scatter plots: are two numeric columns related?

A **scatter plot** draws one point per row, with one numeric column on each axis. Do states with higher violent crime rates also have higher property crime rates?

In [ ]:
ax = sns.scatterplot(data=places_2012, x="Violent", y="Property")
ax.set_title("Violent and property crime rates by state, 2012")
ax.set_xlabel("Violent crimes per 100,000 people")
ax.set_ylabel("Property crimes per 100,000 people")
plt.show()

Each point is one state. The points drift upward from left to right, so states with higher violent crime rates tend to have higher property crime rates. The relationship is far from exact, and one point sits far to the upper right, which is the District of Columbia. A number called the **correlation** summarizes how closely two columns move together. It runs from -1 to 1, where values near 1 mean they rise together, values near -1 mean one falls as the other rises, and values near 0 mean there is no straight-line relationship. `.corr()` calculates it.

In [ ]:
print(places_2012["Violent"].corr(places_2012["Property"]))
without_dc = places_2012[places_2012["State"] != "District of Columbia"]
print(without_dc["Violent"].corr(without_dc["Property"]))

With every state and D.C. the correlation is about 0.67, and without D.C. it drops to about 0.53. One unusual point can noticeably change a correlation, which is a good reason to look at the scatter plot and not only the number. A correlation also says nothing about the cause: neither kind of crime is shown here to produce the other.

The correlation you just saw is **Pearson's r**, the most common kind, which measures how close the points are to a straight line. Very large or very small values can pull it around, as D.C. did. **Spearman's rho** works from the ranks of the values instead of the values themselves, so it asks whether higher in one column goes with higher in the other, and it is less affected by a single extreme point. You choose it with `method="spearman"`. We will not work through either formula.

In [ ]:
print(places_2012["Violent"].corr(places_2012["Property"], method="spearman"))
print(without_dc["Violent"].corr(without_dc["Property"], method="spearman"))

The Spearman correlation only moves from about 0.58 to about 0.55 when D.C. is removed, while Pearson's r above moved from about 0.67 to about 0.53. Calling `.corr()` on a table of numeric columns gives every pair at once.

In [ ]:
places_2012[["Population", "Violent", "Property"]].corr().round(2)

Each column correlates perfectly with itself, which is why the diagonal is all 1s, and the table is the same above and below the diagonal. Population has almost no correlation with either crime rate, at 0.08 and 0.01, which fits what you will see later in this notebook when we color by state size.

## Scatter plots with a regression line

`sns.lmplot()` draws a scatter plot and adds a **regression line**, which is the straight line that best fits the points. The shaded band shows the uncertainty about where that line belongs. `col=` makes a separate chart for each value in a category, which keeps a busy chart readable. Here we add each state's region from a second file. The `merge` step combines two tables, and you do not need to learn it.

In [ ]:
info = pd.read_csv("https://raw.githubusercontent.com/bsheese/CSDS125ExampleData/master/data_state_info.csv")
with_region = places_2012.merge(info, on="State")
grid = sns.lmplot(data=with_region, x="Violent", y="Property", col="Region", col_wrap=2, height=3)
grid.set_axis_labels("Violent crime rate", "Property crime rate")
plt.show()

Each small chart shows one region, and all four use the same scales, so you can compare them directly. The District of Columbia is the point at the far right of the South's chart. It sits just below the line, and no other point is nearby, so the band is wide at that end. D.C. also has a strong pull on the line: without it, the South's line would be steeper, rising about 2.9 property crimes for each additional violent crime instead of about 2.3. The Northeast's line slopes slightly downward. `sns.lmplot()` returns a different kind of object from the other functions, so its labels are set with `grid.set_axis_labels()`.

## Line plots: how does something change over time?

A **line plot** connects points in order, which suits data with a natural sequence such as years. Here is the violent crime rate in Illinois.

In [ ]:
illinois = crime[crime["State"] == "Illinois"]
ax = sns.lineplot(data=illinois, x="Year", y="Violent")
ax.set_title("Violent crime rate in Illinois, 1960-2012")
ax.set_xlabel("Year")
ax.set_ylabel("Violent crimes per 100,000 people")
plt.show()

The rate climbs from 365.1 in 1960 to a peak of 1039.2 in 1991, and then falls to 414.8 by 2012. To compare two lines, make two calls and give each one a `label=`, and then add a legend. Both calls draw on the same chart.

In [ ]:
nation = crime[crime["State"] == "United States"]
ax = sns.lineplot(data=illinois, x="Year", y="Violent", label="Illinois")
sns.lineplot(data=nation, x="Year", y="Violent", label="United States")
ax.set_title("Violent crime rates, 1960-2012")
ax.set_xlabel("Year")
ax.set_ylabel("Violent crimes per 100,000 people")
ax.legend()
plt.show()

Both lines rise and then fall, and the Illinois line stays above the national line in all 53 years. The gap was 281.0 at the 1991 peak and only 27.9 by 2012. You could pull those numbers out of a table, but the chart shows the whole history at once.

If there are several rows for each `x` value, `sns.lineplot()` draws the mean at each `x`, with a shaded band showing the uncertainty around it. Here every state contributes a row for each year.

In [ ]:
all_states = crime[crime["State"] != "United States"]
ax = sns.lineplot(data=all_states, x="Year", y="Violent")
ax.set_title("Mean violent crime rate across states, 1960-2012")
ax.set_xlabel("Year")
ax.set_ylabel("Violent crimes per 100,000 people")
plt.show()

The mean rises to a peak in the early 1990s and then falls. This line averages the states equally, so it is not the same as the national rate, where large states count for more.

## Coloring by a category: `hue`

`hue=` colors the points, lines or bars by a category. This is another way of adding a second category to a chart. The next chart asks whether large states differ from small ones, using a category that we make with a mask.

In [ ]:
places = places_2012.copy()
places["Large"] = places["Population"] > 5000000
ax = sns.scatterplot(data=places, x="Violent", y="Property", hue="Large")
ax.set_title("Crime rates and state size, 2012")
ax.set_xlabel("Violent crimes per 100,000 people")
ax.set_ylabel("Property crimes per 100,000 people")
plt.show()

The two colors are mixed together across the chart, so the size of a state does not seem to separate high-crime states from low-crime ones in this data.

`hue=` also works on a line plot, and it is usually the simpler way to draw several lines when they all come from one table. Earlier in this notebook we made one call for each line and gave each a `label=`. With `hue=`, one call draws a line for each value in the column and adds the legend for you.

In [ ]:
three_states = crime[crime["State"].isin(["Illinois", "Indiana", "Iowa"])]
ax = sns.lineplot(data=three_states, x="Year", y="Violent", hue="State")
ax.set_title("Violent crime rates in three states, 1960-2012")
ax.set_xlabel("Year")
ax.set_ylabel("Violent crimes per 100,000 people")
plt.show()

All three states rise and then fall, but not at the same time. Illinois peaks at 1039.2 in 1991, Iowa at 354.4 in 1995 and Indiana at 537.0 in 1996. Each state has one row per year, so each line shows that state's actual rate, with no shaded band.

## When a chart does not fit

Here is a chart with a real problem. A bar for each of the 51 places, in alphabetical order, cannot be read, and the labels pile on top of each other.

In [ ]:
ax = sns.barplot(data=places_2012, x="State", y="Violent")
ax.set_title("Too many bars, in no useful order")
plt.show()

The fix has three parts: keep only the rows that matter to your question, sort them so that the order means something, and switch the axes so that the names are readable. To make a horizontal bar chart, put the category on `y`.

In [ ]:
top_ten = places_2012.sort_values("Violent", ascending=False).head(10)
ax = sns.barplot(data=top_ten, x="Violent", y="State")
ax.set_title("Ten highest violent crime rates, 2012")
ax.set_xlabel("Violent crimes per 100,000 people")
ax.set_ylabel("")
plt.show()

Now you can read it immediately: the District of Columbia is about twice the rate of Tennessee, which is next.

A horizontal chart is usually the easiest fix for long names, but you may want to keep the categories on the x axis, for example to match other charts in a report. In that case you can tilt the labels with `plt.xticks()`. `rotation=45` turns them 45 degrees, and `ha="right"`, which is short for horizontal alignment, lines up the end of each label with its bar. Call it after the plotting call and before `plt.show()`.

In [ ]:
ax = sns.barplot(data=top_ten, x="State", y="Violent")
ax.set_title("Ten highest violent crime rates, 2012")
ax.set_xlabel("")
ax.set_ylabel("Violent crimes per 100,000 people")
plt.xticks(rotation=45, ha="right")
plt.show()

All ten names can now be read, including long ones such as District of Columbia and South Carolina. Rotating helps with about this many bars, but it would not rescue the chart of 51 places above, which also needs to be cut down and sorted.

Values like the ones in Broadway `Gross` create a different problem, since an axis may show numbers such as `1e9`. The simplest fix is to change the data, by plotting a column in millions or billions.

## Charts that mislead

A chart can be drawn correctly from accurate data and still leave the wrong impression. Two common causes are an axis that does not start at zero and a time range chosen to favor one story. We will start with the axis. Tennessee and Nevada have the second and third highest violent crime rates in the chart above, at 643.6 and 607.6. `ax.set_ylim()` sets the range of the y axis, and here we start it at 600.

In [ ]:
tn_nv = top_ten[top_ten["State"].isin(["Tennessee", "Nevada"])]
ax = sns.barplot(data=tn_nv, x="State", y="Violent")
ax.set_title("A y axis that starts at 600")
ax.set_xlabel("")
ax.set_ylabel("Violent crimes per 100,000 people")
ax.set_ylim(600, 650)
plt.show()

Tennessee's bar looks nearly six times as tall as Nevada's, because the chart shows only the 43.6 and 7.6 above 600. The real rate in Tennessee is only about 6 percent higher. Here is the same data with the axis starting at zero, which is what seaborn does for a bar chart unless you change it.

In [ ]:
ax = sns.barplot(data=tn_nv, x="State", y="Violent")
ax.set_title("The same data with a y axis that starts at zero")
ax.set_xlabel("")
ax.set_ylabel("Violent crimes per 100,000 people")
plt.show()

Now the two bars are close to the same height, which matches the numbers. Readers judge a bar by its length, so a bar chart should normally start at zero. A line plot is different, because readers look at the position and direction of the line, so a line plot often uses a narrower range without causing this problem. Whenever you read a chart, check where its axis starts.

The time range can mislead as well. Here is the Illinois rate again, drawn only from its peak onward.

In [ ]:
since_peak = illinois[illinois["Year"] >= 1991]
ax = sns.lineplot(data=since_peak, x="Year", y="Violent")
ax.set_title("Violent crime rate in Illinois, 1991-2012")
ax.set_xlabel("Year")
ax.set_ylabel("Violent crimes per 100,000 people")
plt.show()

This chart shows a fall of about 60 percent, from 1039.2 to 414.8, and that is accurate. The chart of all 53 years gives a fuller picture: the 2012 rate of 414.8 is still about 14 percent above the 365.1 of 1960. Neither chart is false, but someone who saw only this one could conclude that the rate is now lower than it has ever been. When you choose a range, choose it because of your question, and say in the title which years you are showing.

## Which chart for which question?

| Question | Chart | Function |
|---|---|---|
| What is the shape of one numeric column? | Histogram or box plot | `sns.histplot`, `sns.boxplot` |
| How do categories compare on a number? | Bar chart | `sns.barplot` |
| How many rows are in each category? | Count plot | `sns.countplot` |
| Are two numeric columns related? | Scatter plot | `sns.scatterplot` |
| How does something change over time? | Line plot | `sns.lineplot` |

A chart is finished when someone else could read it, which means it has a title and labeled axes, including units. Saying what it shows is part of the job too. A useful description names the pattern and gives numbers, as in "the Illinois rate peaked at 1039.2 in 1991 and fell to 414.8 by 2012."

## Reusing a chart with a function

If you are making the same chart for several states, a function saves you from repeating yourself, as functions did in module 04. This one takes the name of a state, and it does not need anything new.

In [ ]:
def plot_state(state_name):
    rows = crime[crime["State"] == state_name]
    ax = sns.lineplot(data=rows, x="Year", y="Violent")
    ax.set_title("Violent crime rate in " + state_name)
    ax.set_xlabel("Year")
    ax.set_ylabel("Violent crimes per 100,000 people")
    plt.show()

plot_state("Vermont")

Vermont's rate starts at 9.5 in 1960, peaks at 184.2 in 1979 and is 142.6 by 2012. Its peak comes about 12 years before the 1991 peak in Illinois, which is the kind of difference you can spot quickly when a chart for another state takes only one more line of code.

## Practice problems

These problems are not collected or graded. They are here so that you can find out whether you can write this code yourself before the reading check and the pair activity. Write your answer in the cell under each problem, run it, and then run the check cell below it, which tells you whether your answer is right. The check cells are collapsed because they contain the answers, so try the problem before you open one.

Run the next two cells first. The first one creates fresh data for the problems, and the second sets up the checks.

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

crime = pd.read_csv("https://raw.githubusercontent.com/bsheese/CSDS125ExampleData/master/data_corgis_state_crime.csv")
crime = crime.rename(columns={"Rates.Violent.All": "Violent", "Rates.Property.All": "Property"})
crime = crime[["State", "Year", "Population", "Violent", "Property"]]
places_2012 = crime[(crime["Year"] == 2012) & (crime["State"] != "United States")]
crime_start = crime.copy()
illinois_rows = crime[crime["State"] == "Illinois"]
indiana_rows = crime[crime["State"] == "Indiana"]
three_rows = crime[crime["State"].isin(["Illinois", "Indiana", "Iowa"])]
two_states_2012 = places_2012[places_2012["State"].isin(["Illinois", "Michigan"])]
top_five_want = places_2012.sort_values("Property", ascending=False).head(5)["Property"].to_list()

In [ ]:
#@title Run this cell to set up the checks
import numpy as np
import pandas as pd


def same(got, want):
    """True when got matches want: same labels and values for a series, same value for a number."""
    try:
        if isinstance(want, pd.DataFrame):
            if not isinstance(got, pd.DataFrame) or list(got.columns) != list(want.columns) or len(got) != len(want):
                return False
            if not pd.api.types.is_integer_dtype(want.index) and list(got.index) != list(want.index):
                return False
            got, want = got.reset_index(drop=True), want.reset_index(drop=True)
            for column in want.columns:
                if pd.api.types.is_numeric_dtype(want[column]):
                    if not np.allclose(pd.to_numeric(got[column], errors="coerce").astype(float), want[column].astype(float), equal_nan=True):
                        return False
                elif got[column].fillna("").astype(str).to_list() != want[column].fillna("").astype(str).to_list():
                    return False
            return True
        if isinstance(want, pd.Series):
            if not isinstance(got, pd.Series) or list(got.index) != list(want.index):
                return False
            if pd.api.types.is_numeric_dtype(want):
                return bool(np.allclose(got.astype(float), want.astype(float), equal_nan=True))
            return list(got) == list(want)
        if isinstance(want, (list, np.ndarray)):
            return list(got) == list(want)
        if isinstance(want, str):
            return got == want
        if isinstance(got, (pd.Series, pd.DataFrame, list, np.ndarray)):
            return False
        return bool(np.isclose(float(got), float(want)))
    except (TypeError, ValueError):
        return False


def check(number, name, want, hint, unchanged=None):
    """Print feedback for one problem. unchanged is (variable name, its starting value)."""
    if name not in globals():
        print(f"Problem {number}: not yet. No variable named {name} exists. Did you run your answer cell?")
    elif not same(globals()[name], want):
        print(f"Problem {number}: not yet. {hint}")
    elif unchanged is not None and not globals()[unchanged[0]].equals(unchanged[1]):
        print(f"Problem {number}: {name} is right, but {unchanged[0]} changed. Work on a copy, then run the setup cell again.")
    else:
        print(f"Problem {number}: correct.")


def check_chart(number, name, test, hint, y_label=True):
    """Check a chart stored in a variable: it has labels, and test(chart) is True."""
    if name not in globals():
        print(f"Problem {number}: not yet. No variable named {name} exists. Did you run your answer cell?")
        return
    chart = globals()[name]
    if not hasattr(chart, "get_title"):
        print(f"Problem {number}: not yet. Store what the seaborn function returns in {name}.")
    elif not chart.get_title() or not chart.get_xlabel() or (y_label and not chart.get_ylabel()):
        print(f"Problem {number}: not yet. Give the chart a title and label its axes.")
    else:
        try:
            ok = bool(test(chart))
        except (AttributeError, IndexError, TypeError, ValueError):
            ok = False
        print(f"Problem {number}: correct." if ok else f"Problem {number}: not yet. {hint}")

### Problem 1
Make a scatter plot with `Population` on the x axis and `Property` on the y axis, using `places_2012`, and store it in `pop_scatter`. Give it a title and label both axes.

In [ ]:
# Problem 1

In [ ]:
#@title Check Problem 1
check_chart(1, "pop_scatter", lambda c: len(c.collections[0].get_offsets()) == 51, "Use sns.scatterplot() with data=places_2012.")

### Problem 2
Store the Pearson correlation between `Population` and `Property` in `places_2012` in a variable named `pop_corr`.

In [ ]:
# Problem 2

In [ ]:
#@title Check Problem 2
check(2, "pop_corr", places_2012["Population"].corr(places_2012["Property"]), "Call .corr() on one column and pass it the other.")

### Problem 3
Store the Spearman correlation between the same two columns in a variable named `pop_spearman`.

In [ ]:
# Problem 3

In [ ]:
#@title Check Problem 3
check(3, "pop_spearman", places_2012["Population"].corr(places_2012["Property"], method="spearman"), "Add method=\"spearman\".")

### Problem 4
Make a line plot of `Property` by `Year` using `illinois_rows`, and store it in `il_line`. Give it a title and label both axes.

In [ ]:
# Problem 4

In [ ]:
#@title Check Problem 4
check_chart(4, "il_line", lambda c: len(c.get_lines()[0].get_xdata()) == 53, "Use sns.lineplot() with data=illinois_rows.")

### Problem 5
Draw two lines of `Property` by `Year` on one chart, one from `illinois_rows` and one from `indiana_rows`, with a legend. Store the chart in `two_lines`. Give it a title and label both axes.

In [ ]:
# Problem 5

In [ ]:
#@title Check Problem 5
check_chart(5, "two_lines", lambda c: len(c.get_lines()) >= 2 and c.get_legend() is not None, "Make two lineplot calls, give each a label=, and call .legend().")

### Problem 6
Make a horizontal bar chart of the five places in `places_2012` with the highest `Property` rates, ordered from highest to lowest, and store it in `top_bars`. Give it a title and an x axis label.

In [ ]:
# Problem 6

In [ ]:
#@title Check Problem 6
check_chart(6, "top_bars", lambda c: np.allclose([p.get_width() for p in c.patches if p.get_width() > 0], top_five_want), "Sort, take the first five, and put State on y.", y_label=False)

### Problem 7
Using `three_rows`, which holds Illinois, Indiana and Iowa, make a line plot of `Property` by `Year` with one line per state. Use one `sns.lineplot()` call with `hue=`, and store the chart in `hue_lines`. Give it a title and label both axes.

In [ ]:
# Problem 7

In [ ]:
#@title Check Problem 7
check_chart(7, "hue_lines", lambda c: {t.get_text() for t in c.get_legend().get_texts()} == {"Illinois", "Indiana", "Iowa"}, "Use sns.lineplot() with data=three_rows and hue=\"State\".")

### Problem 8
Make a vertical bar chart of the same five places from Problem 6, with `State` on the x axis, and tilt the labels 45 degrees. Store the chart in `tilted_bars`. Give it a title and a y axis label.

In [ ]:
# Problem 8

In [ ]:
#@title Check Problem 8
check_chart(8, "tilted_bars", lambda c: np.allclose([p.get_height() for p in c.patches if p.get_height() > 0], top_five_want) and all(t.get_rotation() == 45 for t in c.get_xticklabels()), "Put State on x and Property on y, then add plt.xticks(rotation=45, ha=\"right\") before plt.show().")

### Problem 9
The cell below already has code that compares property crime in Illinois and Michigan. Run it and look at the chart. It makes the gap look large, but the real rates are 2578.7 and 2530.5, so Illinois is only about 2 percent higher. Fix the code so that the y axis starts at zero, and change the title so that it no longer claims a large difference. Keep the chart stored in `fair_bars`.

In [ ]:
# Problem 9
fair_bars = sns.barplot(data=two_states_2012, x="State", y="Property")
fair_bars.set_title("Illinois has far more property crime than Michigan, 2012")
fair_bars.set_xlabel("State")
fair_bars.set_ylabel("Property crimes per 100,000 people")
fair_bars.set_ylim(2500, 2600)
plt.show()

In [ ]:
#@title Check Problem 9
check_chart(9, "fair_bars", lambda c: c.get_ylim()[0] == 0 and "far more" not in c.get_title().lower() and np.allclose(sorted(p.get_height() for p in c.patches if p.get_height() > 0), [2530.5, 2578.7]), "Remove the set_ylim() line or start the axis at 0, and write a title that does not say far more.")

### Problem 10
The cell below already has code that draws Indiana's violent crime rate. Run it and look at the chart. The rate really did rise from 323.3 in 2010 to 345.7 in 2012, but the chart shows only those three years and its title makes them sound like the whole story. Fix the code so that it uses all of `indiana_rows`, and change the title so that it names the years shown, 1960 and 2012, instead of making a claim. Keep the chart stored in `full_line`.

In [ ]:
# Problem 10
recent = indiana_rows[indiana_rows["Year"] >= 2010]
full_line = sns.lineplot(data=recent, x="Year", y="Violent")
full_line.set_title("Violent crime is climbing fast in Indiana")
full_line.set_xlabel("Year")
full_line.set_ylabel("Violent crimes per 100,000 people")
plt.show()

In [ ]:
#@title Check Problem 10
check_chart(10, "full_line", lambda c: len(c.get_lines()[0].get_xdata()) == 53 and "1960" in c.get_title() and "2012" in c.get_title(), "Use data=indiana_rows with no year filter, and put 1960 and 2012 in the title.")

## What's next

That finishes the notebooks for the course. You can now load a dataset, check it, select from it, summarize it, compare groups and show your results in a chart. These are the core skills of working with data, and CS/DS 225 builds on them with more detailed work on cleaning, grouping and visualization.